# Data Preparation

Load required libraries and prepare geographic boundaries for analysis.

In [2]:
import contextily as ctx
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import geopandas as gpd
import seaborn as sns
import warnings
import zipfile

from IPython.display import display
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import ListedColormap
from matplotlib.patches import Circle, RegularPolygon
from matplotlib.path import Path
from matplotlib.projections.polar import PolarAxes
from matplotlib.projections import register_projection
from matplotlib.spines import Spine
from matplotlib.transforms import Affine2D
from sklearn.cluster import KMeans
from sklearn import preprocessing
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import PartialDependenceDisplay
from sklearn.model_selection import train_test_split
from sklearn import metrics, preprocessing

# Load Tiger/Line shapefiles

Load census tract boundaries for 2019-2024 while dropping Catalina Island tract GEOIDs

In [5]:
drop_geoids = ['06037990100', '06037990200', '06037990300']  # Catalina Island tracts

for year in range(2019, 2025):
    path = f"Data/tl_shp_library/tl_{year}_06_tract.zip"
    tracts = gpd.read_file(path)
    la_tracts = tracts[(tracts["COUNTYFP"] == "037") & (tracts["STATEFP"] == "06")]
    la_tracts = la_tracts.set_index('GEOID')
    la_tracts = la_tracts.drop(index=drop_geoids, errors='ignore')
    
    var_name = f"tracts{str(year)[-2:]}"
    globals()[var_name] = la_tracts

# Save
pd.to_pickle(tracts19, "data/tl_shp_library/tracts19.pkl")
pd.to_pickle(tracts23, "data/tl_shp_library/tracts23.pkl")


# Land Use Analysis

Analyze land use patterns to understand residential concentration and parking availability.

In [7]:
# Process land use data from raw file
with zipfile.ZipFile("Data/landuse.gpkg.zip", 'r') as zip_ref:
    zip_ref.extractall("Data/")
luGdf = gpd.read_file("Data/landuse.gpkg")

# Select relevant columns
selected_cols = ['ACRES', 'GEOID20', 'DEN_AVG', 'ZN19_SCAG', 'LU19', 'geometry']
luGdf = luGdf[selected_cols]

# Create major land use categories
luGdf['LU19'] = pd.to_numeric(luGdf['LU19'], errors='coerce')
lu19_major_categories = {
    1100: "Residential",
    1200: "Commercial and Services",
    1300: "Industrial",
    1400: "Transportation, Communications, and Utilities",
    1500: "Public Facilities",
    1600: "Open Space and Recreation",
    1700: "Vacant",
    1800: "Water",
    1900: "Agriculture",
    8888: "Undevelopable or Protected Land",
    9999: "LandUnknown"
}

luGdf['LU19_major_code'] = (luGdf['LU19'] // 100) * 100
luGdf.loc[luGdf['LU19'] == 8888, 'LU19_major_code'] = 8888
luGdf.loc[luGdf['LU19'] == 9999, 'LU19_major_code'] = 9999
luGdf['LU19_major_label'] = luGdf['LU19_major_code'].map(lu19_major_categories)

# Save processed land use data
luGdf.to_file("Data/landuse_processed.gpkg", driver="GPKG")

/opt/anaconda3/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: Non-conformant content for record 1 in column YEAR_GP, 2015-05-01T00:00:00.0Z, successfully parsed
  return ogr_read(


# Amenities Analysis

Count public facilities and parking within buffer zones around census tracts.

In [8]:
# Create 2km buffers around tracts
tracts23_buffer = tracts23.to_crs(epsg=3497).copy()
tracts23_buffer["geometry"] = tracts23_buffer.buffer(2000)

# Load public facilities data
facilities = {
    'ArtsRecreation': gpd.read_file("data/public_facilities/ArtsRecreation.gpkg").to_crs(epsg=3497),
    'Education': gpd.read_file("data/public_facilities/Education.gpkg").to_crs(epsg=3497),
    'Hospitals': gpd.read_file("data/public_facilities/Hospitals.gpkg").to_crs(epsg=3497),
    'MunicipalServices': gpd.read_file("data/public_facilities/MunicipalServices.gpkg").to_crs(epsg=3497),
    'PhysicalFeatures': gpd.read_file("data/public_facilities/PhysicalFeatures.gpkg").to_crs(epsg=3497),
    'PublicSafety': gpd.read_file("data/public_facilities/PublicSafety.gpkg").to_crs(epsg=3497),
    'Transportation': gpd.read_file("data/public_facilities/Transportation.gpkg").to_crs(epsg=3497)
}

# Remove jails from public safety facilities
facilities['PublicSafety'] = facilities['PublicSafety'][
    ~facilities['PublicSafety']["cat2"].isin(["Jails and Prisons", "Probation Camps and Juvenile Halls"])
].reset_index(drop=True)

# Load food pantries
pantry_df = pd.read_csv('data/Food_Resources_in_California.csv')
pantry_df = pantry_df[pantry_df.County == 'Los Angeles']
facilities['Pantry'] = gpd.GeoDataFrame(
    pantry_df, 
    geometry=gpd.points_from_xy(pantry_df.Longitude, pantry_df.Latitude, crs='EPSG:4326')
).to_crs(epsg=3497)

# Extract parking lots from land use data
parking = luGdf[luGdf['LU19'] == 1247].to_crs(epsg=3497).copy()
parking = (parking.rename(columns={'GEOID20': 'GEOID'}).set_index('GEOID'))
parking['geometry'] = parking.centroid

# Save processed amenities data
tracts23_buffer.to_pickle("data/tl_shp_library/tracts23_buffer.pkl")
pd.to_pickle(facilities, "data/public_facilities/facilities_dict.pkl")
parking.to_pickle("data/public_facilities/parking_centroids.pkl")

/opt/anaconda3/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: Non-conformant content for record 1 in column last_update, 2025-03-10T00:00:00.0Z, successfully parsed
  return ogr_read(
/opt/anaconda3/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: Non-conformant content for record 177 in column last_update, 2025-04-04T13:03:56.0Z, successfully parsed
  return ogr_read(


# Utility functions for spatial analysis

Natural Breaks (Jenks) and quantile function

In [3]:
%%writefile jenks_utils.py

import numpy as np
import pandas as pd

def jenks_breaks(data, k=5):
    """Calculate Jenks Natural Breaks classification"""
    data = sorted(data)
    n = len(data)
    mat1, mat2 = np.zeros((n+1, k+1)), np.zeros((n+1, k+1))
    
    for i in range(1, k+1):
        mat1[1][i] = 1
        for j in range(2, n+1):
            mat2[j][i] = float('inf')
    
    for i in range(2, n+1):
        s1 = s2 = w = 0
        for j in range(1, i+1):
            val = data[i-j]
            s1 += val
            s2 += val * val
            w += 1
            variance = s2 - (s1 * s1) / w
            if i-j > 0:
                for m in range(2, k+1):
                    if mat2[i][m] >= variance + mat2[i-j][m-1]:
                        mat1[i][m] = i-j+1
                        mat2[i][m] = variance + mat2[i-j][m-1]
        mat1[i][1] = 1
        mat2[i][1] = s2 - (s1 * s1) / w
    
    breaks = []
    k_idx = n
    for i in range(k, 0, -1):
        breaks.append(data[int(mat1[k_idx][i])-1])
        k_idx = int(mat1[k_idx][i]-1)
    
    return sorted(set(breaks))

Overwriting jenks_utils.py


# Natural Language Processing

In [ ]:
# pull reddit data

import praw
import pandas as pd
import os

client_id = '#'
client_secret = '#'
user_agent = 'first scrape project'

reddit = praw.Reddit(
    client_id=client_id,
    client_secret=client_secret,
    user_agent=user_agent,
)

def get_reddit_and_export(subreddit_name):
    r_list = []
    for submission in reddit.subreddit(subreddit_name).new(limit=500):
        r_list.append(submission.title)
        
        # Replace all MoreComments with actual comments
        submission.comments.replace_more(limit=0)
        
        # Only get comments that have a body attribute
        r_list += [c.body for c in submission.comments.list() 
                  if hasattr(c, 'body') and c.body not in ['[deleted]', '[removed]']]
        
    print('Retrieved {} comments for {}'.format(len(r_list), subreddit_name))
    
    # Export to CSV in Data/reddit folder
    filepath = f'Data/reddit/{subreddit_name}.csv'
    pd.DataFrame(r_list).to_csv(filepath, index=False)
    print(f'Exported {filepath}')
    
    return r_list

urbancarliving = get_reddit_and_export('urbancarliving')

In [4]:
# Topic modeling data prep

import pandas as pd
import re
import gensim
import pyLDAvis
import pyLDAvis.gensim_models
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from collections import Counter
import matplotlib.pyplot as plt

import nltk
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /Users/admin/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /Users/admin/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
# Load and examine data
df = pd.read_csv('Data/reddit/urbancarliving.csv')
print(f"Loaded {len(df)} comments")
print(f"Columns: {df.columns.tolist()}")
print(f"First few comments:")
for i in range(3):
    print(f"{i+1}: {df.iloc[i, 0][:100]}...")

# Clean and prepare text
def clean_text(text):
    if not isinstance(text, str) or text in ['[deleted]', '[removed]', '']:
        return ""
    
    # Remove URLs, mentions, hashtags
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+|#\w+', '', text)
    
    # Remove punctuation, numbers, etc.
    text = re.sub(r"[^A-Za-z\s]", "", text)
    
    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()

# Apply cleaning
comments = df.iloc[:, 0].apply(clean_text)
comments = comments[comments.str.len() > 10]  # Remove very short comments
print(f"After cleaning: {len(comments)} comments")

# Tokenize and remove stopwords
swords = stopwords.words('english')

# Remove punctuation from stopwords to match your cleaned text
swords_clean = []
for word in swords:
    # Apply same cleaning as clean_text function
    clean_word = re.sub(r"[^A-Za-z\s]", "", word)
    if clean_word and len(clean_word) > 0:
        swords_clean.append(clean_word.lower())

# Remove duplicates and empty strings
swords_clean = list(set([word for word in swords_clean if word]))
swords_clean = sorted(swords_clean)

print(f"Original stopwords: {len(swords)}")
# print(swords)
print(f"Cleaned stopwords: {len(swords_clean)}")
# print(swords_clean)

# Add specific stopwords
reddit_stopwords = ['reddit', 'edit', 'update', 'deleted', 'removed', 'comment', 'post', 'thread']
additional_stopwords = ['lets', 'thats', 'people', 'even', 'time', 'way', 'thats', 'youre', 'much', 
                        'would', 'could', 'get', 'one', 'also', 'really', 'think', 'know', 'like']

swords_clean.extend(reddit_stopwords + additional_stopwords)

print(f"Extended stopwords: {len(swords_clean)}")
# print(swords_clean)

# Tokenize
wordlists = []
for comment in comments:
    words = [word for word in word_tokenize(comment.lower()) 
            if word not in swords_clean 
            and len(word) > 0
            and word.isalpha()]
    
    wordlists.append(words)

print(f"Prepared {len(wordlists)} comments for modeling")

# Save as pickle
pd.to_pickle(wordlists, "data/reddit/wordlists.pkl")
pd.to_pickle(comments, "data/reddit/comments-urbancarliving.pkl")